# CNN Autoencoder for Anomaly Detection Using Toy Car Audio

**Purpose:** This notebook presents a complete pipeline for toy car anomaly detection based on operating sound, including data loading, preprocessing, CNN autoencoder training, and result analysis.

## 1. Imports

In [ ]:
import os 
import glob 
import random
from collections import defaultdict, Counter

import numpy as np

import librosa
import librosa.display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Subset, DataLoader

from sklearn.metrics import (
    f1_score, 
    precision_recall_curve, 
    average_precision_score 
)

import pickle

import optuna
import optuna.visualization as vis

import matplotlib.pyplot as plt

from concurrent.futures import ThreadPoolExecutor, as_completed

## 2. Paths and configuration

In [ ]:
# Paths

DATA_BASE_PATH = "F:ToyCar"

BEST_MODEL_SAVE_PATH = "best_Transformer_AE.pth"

In [ ]:
# Configs

CASES = ['case1', 'case2', 'case3', 'case4']
CHANNELS = ['1', '2', '3', '4']
SEED = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

## 3. Data Loading

In [ ]:
def load_all_wav_paths(data_base_path: str, cases: list[str], folder: str="NormalSound_IND") -> list[str]:
    """
    Get the paths of the .wav files from the folder
    """
    all_wavs = []
    for case in cases:
        path = f'{data_base_path}/{case}/{folder}'
        all_wavs += glob.glob(os.path.join(path, '*.wav'))
    return all_wavs

def structure_wav_paths(wav_paths: list[str]) -> defaultdict:
    """
    Structure the wav file path according to case number, channel and group id
    Example:
        1100041309_ToyCar_case1_normal_IND_ch4_1309.wav -> data[case][sample_id][ch] = data['case1']['1309']['4']

    Returns:
    paths = {
        "case1": {
            "sample_1": /path/to/the.wav,
            "sample_2": ...
        }
    }
    """
    paths = defaultdict(lambda: defaultdict(dict))

    for path in wav_paths:
        parts = path.replace('.wav', '').split('_')[1:] # retain file name only
        case = parts[2]
        channel = parts[-2].replace('ch', '')
        sample_id = parts[-1]
        paths[case][sample_id][channel] = path

    return paths

In [ ]:
# Utils
def crop_audio(audio, sr, start_sec=None, end_sec=None):
    if start_sec is None or end_sec is None:
        return audio

    start = int(start_sec * sr)
    end = int(end_sec * sr)

    return audio[max(0, start):min(len(audio), end)]


def normalize(spec):
    return (spec - spec.min()) / (spec.max() - spec.min() + 1e-8)


def wav_to_logmel(audio, sr):
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_fft=1024,
        hop_length=512,
        n_mels=64
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    return normalize(log_mel)

In [ ]:
# Normal Audio Dataset
class NormalDataset(Dataset):
    def __init__(self, data_base_path, cases, start_sec=None, end_sec=None, verbose=True):
        self.samples = []
        self.start_sec = start_sec
        self.end_sec = end_sec
        self.sample_keys = []
        structured = {}

        for case in cases:
            path = f"{data_base_path}/{case}/NormalSound_IND"
            wavs = glob.glob(os.path.join(path, "*.wav"))

            for wav in wavs:
                parts = os.path.basename(wav).replace(".wav", "").split("_")

                case_name = parts[2]
                ch = parts[-2].replace("ch", "")
                sample_id = parts[-1]

                key = (case_name, sample_id)

                if key not in structured:
                    structured[key] = {}

                structured[key][ch] = wav

        # keep only full 4-channel samples
        for key, channels in structured.items():
            if len(channels) == len(CHANNELS):
                case_name, sample_id = key  
                self.sample_keys.append((case_name, sample_id))
                self.samples.append(channels)
            elif verbose:
                print(f"[WARN] Skipping incomplete {key}")

        if verbose:
            print(f"[INFO] Normal samples: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        channels = self.samples[idx]

        mel_channels = []

        for ch in CHANNELS:
            try:
                audio, sr = librosa.load(channels[ch], sr=None)
            except Exception as e:
                # skip if failed loading
                print(f"[ERROR] Failed loading {channels[ch]}: {e}")
                return self.__getitem__((idx + 1) % len(self))  
            audio = crop_audio(audio, sr, self.start_sec, self.end_sec)
            mel = wav_to_logmel(audio, sr)
            mel_channels.append(mel)

        # align time
        min_T = min(m.shape[1] for m in mel_channels)
        mel_channels = [m[:, :min_T] for m in mel_channels]

        x = np.stack(mel_channels)  # shape: (4, 64, T)
        return torch.tensor(x, dtype=torch.float32)

In [ ]:
class AnomalousDataset(Dataset):
    def __init__(self, data_base_path, cases, start_sec=None, end_sec=None, verbose=True):
        self.samples = []
        self.start_sec = start_sec
        self.end_sec = end_sec

        structured = {}
        self.sample_keys = []

        for case in cases:
            path = f"{data_base_path}/{case}/AnomalousSound_IND"
            wavs = glob.glob(os.path.join(path, "*.wav"))

            for wav in wavs:
                parts = os.path.basename(wav).replace(".wav", "").split("_")

                case_name = parts[2]
                condition = parts[3]
                suffix = parts[-1]
                ch = parts[-2].replace("ch", "")

                sample_id = f"{condition}_{suffix}"

                key = (case_name, sample_id)

                if key not in structured:
                    structured[key] = {}

                structured[key][ch] = wav

        # keep only valid 4-channel groups
        for key, channels in structured.items():
            if all(ch in channels for ch in CHANNELS):
                case_name, sample_id = key
                self.samples.append(channels)
                self.sample_keys.append((case_name, sample_id))
            elif verbose:
                print(f"[WARN] skipping incomplete anomaly {key}")

        if verbose:
            print(f"[INFO] Anomalous grouped samples: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        channels = self.samples[idx]

        mel_channels = []
        for ch in ['1','2','3','4']:
            try:
                audio, sr = librosa.load(channels[ch], sr=None)
            except Exception as e:
                print(f"[ERROR] Failed loading {channels[ch]}: {e}")
                return self.__getitem__((idx + 1) % len(self))  
            audio = crop_audio(audio, sr, self.start_sec, self.end_sec)
            mel = wav_to_logmel(audio, sr)
            mel_channels.append(mel)

        min_T = min(m.shape[1] for m in mel_channels)
        mel_channels = [m[:, :min_T] for m in mel_channels]

        return torch.tensor(np.stack(mel_channels), dtype=torch.float32)

In [ ]:
normal_dataset = NormalDataset(
    DATA_BASE_PATH,
    cases=CASES,
    start_sec=1.5,
    end_sec=7.7
)

anom_dataset = AnomalousDataset(
    DATA_BASE_PATH,
    cases=CASES,
    start_sec=1.5,
    end_sec=7.7
)

print(len(anom_dataset))
print(len(normal_dataset))

## 4. Data Proprocessing

In [ ]:
def convert_to_logmel(structured_paths: defaultdict) -> defaultdict:
    """
    Converts and loads wav paths into log-mel data
    .wav path -> mel_data[case][sample_id] = size[ch, mel_bins, time]

    mel_data = {
        "case1": {
            "sample_1": np.array([4, mel, time]),
            "sample_2": ...
        }
    }
    """
    mel_data = defaultdict(dict)
    total_count = sum(len(samples)*4 for samples in structured_paths.values()) # 4 channels for each sample
    count = 0

    for case in structured_paths: 
        for sample_id, channels in structured_paths[case].items(): 
            if len(channels) != len(CHANNELS):
                # skip sample with incomplete channels (< 4 channels)
                print(f"[Warning]: Skipping incomplete sample (case: {case}, sample_id: {sample_id}) \
                      with channel size {len(channels)} instead of {len(CHANNELS)}.")
                continue
        
            mel_channels = []
            success = True

            for ch in CHANNELS:
                try:
                    # sr = sample rate, it is set to None here for auto detection (48000Hz)
                    audio, sr = librosa.load(channels[ch], sr=None)
                    success = True
                    
                    count += 1
                    if count % (total_count // 20) == 0:
                        print(f"[progress] {count}/{total_count}")
                except:
                    print(f"[Error]: wav loading error, skipping {channels[ch]}")
                    success = False
                    break

                mel = librosa.feature.melspectrogram(
                    y=audio, 
                    sr=sr, 
                    n_fft=1024,     # window size
                    hop_length=512, # number of steps to next window
                    n_mels=64       # number of mel bins (filter banks)
                )

                log_mel = np.log(mel + 1e-8)
                mel_channels.append(log_mel)
            
            if success:
                mel_data[case][sample_id] = np.stack(mel_channels) # (n_channels, n_mels, n_frames)
            
    return mel_data

def load_file(args):
    case, file_path, sample_id = args
    print(file_path)
    return case, sample_id, np.load(file_path, mmap_mode='r')

def load_precomputed_samples(folder):

    mel_data = defaultdict(dict)
    tasks = []

    for case in os.listdir(folder)[:1]:
        case_path = os.path.join(folder, case)
        if not os.path.isdir(case_path):
            continue

        for file in os.listdir(case_path):
            if file.endswith(".npy"):
                sample_id = file[:-4]
                file_path = os.path.join(case_path, file)
                tasks.append((case, file_path, sample_id))

    with ThreadPoolExecutor(max_workers=16) as executor:
        for case, sample_id, mel in executor.map(load_file, tasks):
            mel_data[case][sample_id] = mel

    return mel_data

def reduce_samples(structured_samples, max_per_case=200):
    reduced = {}

    for case in structured_samples:
        samples = list(structured_samples[case].items())
        
        # random subset
        selected = random.sample(samples, min(max_per_case, len(samples)))
        
        reduced[case] = dict(selected)
    
    return reduced

def sec_to_frame(sec, sr, hop_length=512):
    return int(sec * sr / hop_length)

def crop_mel(mel, start_sec, end_sec, sr=48000, hop_length=512):
    start_f = int(start_sec * sr / hop_length)
    end_f = int(end_sec * sr / hop_length)

    start_f = max(0, start_f)
    end_f = min(mel.shape[-1], end_f)

    return mel[:, :, start_f:end_f]   # (channels, mel, time)

In [ ]:
# =========================
# 1. Define directories
# =========================

cnt_root = os.path.join(data_base_path, "npy", "CNT")
ind_root = os.path.join(data_base_path, "npy", "IND")
cnt_seg_root = os.path.join(data_base_path, "npy", "CNT_SEG")
os.makedirs(cnt_seg_root, exist_ok=True)

# =========================
# 2. Detect 10-sec length
# =========================
sample_ind_files = glob.glob(os.path.join(ind_root, "*", "normal", "*.npy"))

if not sample_ind_files:
    raise FileNotFoundError("No IND normal files found.")

sample_ind_data = np.load(sample_ind_files[0])
time_axis = int(np.argmax(sample_ind_data.shape))
frames_per_10_sec = sample_ind_data.shape[time_axis]

print(f"[INFO] 10 seconds = {frames_per_10_sec} frames")

# =========================
# 3. Worker function
# =========================
def process_file(file_path):
    try:
        base_name = os.path.basename(file_path).replace(".npy", "")

        if "_seg" in base_name:
            return None

        spectrogram = np.load(file_path)
        total_frames = spectrogram.shape[time_axis]

        num_full_chunks = total_frames // frames_per_10_sec
        remainder = total_frames % frames_per_10_sec

        case_name = os.path.basename(os.path.dirname(file_path))
        case_output_dir = os.path.join(cnt_seg_root, case_name)
        os.makedirs(case_output_dir, exist_ok=True)

        saved = 0

        for i in range(num_full_chunks):
            start = i * frames_per_10_sec
            end = start + frames_per_10_sec

            slices = [slice(None)] * spectrogram.ndim
            slices[time_axis] = slice(start, end)
            segment = spectrogram[tuple(slices)]

            output_path = os.path.join(
                case_output_dir, f"{base_name}_seg{i:02d}.npy"
            )
            np.save(output_path, segment)
            saved += 1

        return (case_name, base_name, saved, remainder)

    except Exception as e:
        print(f"[ERROR] {file_path}: {e}")
        return None

# =========================
# 4. Run (Jupyter-safe)
# =========================
cnt_files = glob.glob(os.path.join(cnt_root, "*", "*.npy"))
print(f"[INFO] Found {len(cnt_files)} CNT files")

max_workers = 8  # safe for I/O

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(process_file, f) for f in cnt_files]

    for future in as_completed(futures):
        result = future.result()
        if result is None:
            continue

        case_name, base_name, saved, remainder = result

        if remainder > 0:
            print(f"  -> {case_name}/{base_name}: {saved} chunks, discarded {remainder}")
        else:
            print(f"  -> {case_name}/{base_name}: {saved} chunks")

print(f"[SUCCESS] Done. Saved in {cnt_seg_root}")

In [ ]:
# .npy dataset for both normal and anomalous preprocessed data
class UnifiedNPYDataset(Dataset):
    def __init__(
        self,
        base_path,
        target_T,
        cases=("case1",),
        use_ind=True,
        use_cnt=True,
        include_anomaly=True,
        stride_ratio=5.0,   # if 0.5, then 50% overlap
    ):
        self.samples = []  # (path, start, label)
        self.target_T = target_T
        self.stride = int(target_T * stride_ratio)
        self.rng = np.random.default_rng(42)

        ind_normal_samples = []
        ind_anom_samples = []
        cnt_candidates = []

        dropped_short = 0

        for case in cases:

            # IND 
            if use_ind:
                normal_dir = os.path.join(base_path, "IND", case, "normal")
                for p in glob.glob(os.path.join(normal_dir, "*.npy")):
                    x = np.load(p, mmap_mode="r")
                    T = x.shape[-1]

                    if T < target_T:
                        dropped_short += 1
                        continue

                    ind_normal_samples.append((p, None, 0))

            if use_ind and include_anomaly:
                anom_dir = os.path.join(base_path, "IND", case, "anomaly")
                for p in glob.glob(os.path.join(anom_dir, "*.npy")):
                    x = np.load(p, mmap_mode="r")
                    T = x.shape[-1]

                    if T < target_T:
                        dropped_short += 1
                        continue

                    ind_anom_samples.append((p, None, 1))

            # CNT (using sliding windows)
            if use_cnt:
                cnt_seg_dir = os.path.join(base_path, "CNT_SEG", case)
                cnt_dir = os.path.join(base_path, "CNT", case)

                if os.path.isdir(cnt_seg_dir):
                    cnt_paths = glob.glob(os.path.join(cnt_seg_dir, "*.npy"))
                else:
                    seg_paths = glob.glob(os.path.join(cnt_dir, "*_seg*.npy"))
                    cnt_paths = seg_paths if len(seg_paths) > 0 else glob.glob(os.path.join(cnt_dir, "*.npy"))

                for p in cnt_paths:
                    x = np.load(p, mmap_mode="r")
                    T = x.shape[-1]

                    if T < target_T:
                        dropped_short += 1
                        continue

                    # generate sliding windows
                    for start in range(0, T - target_T + 1, self.stride):
                        cnt_candidates.append((p, start, 0))

        # Balance CNT to IND normal
        if use_ind:
            self.samples.extend(ind_normal_samples)

        sampled_cnt = []
        if use_cnt:
            target_cnt = len(ind_normal_samples) if use_ind else len(cnt_candidates)

            if target_cnt > 0 and len(cnt_candidates) > 0:
                if len(cnt_candidates) >= target_cnt:
                    pick_idx = self.rng.choice(len(cnt_candidates), size=target_cnt, replace=False)
                    sampled_cnt = [cnt_candidates[i] for i in pick_idx]
                else:
                    sampled_cnt = cnt_candidates
                    print(f"[WARN] CNT candidates ({len(cnt_candidates)}) < target ({target_cnt})")

                self.samples.extend(sampled_cnt)

        if use_ind and include_anomaly:
            self.samples.extend(ind_anom_samples)

        print(f"[INFO] IND normal: {len(ind_normal_samples)}")
        print(f"[INFO] CNT candidates (windows): {len(cnt_candidates)}")
        print(f"[INFO] CNT sampled: {len(sampled_cnt)}")
        print(f"[INFO] IND anomaly: {len(ind_anom_samples)}")
        print(f"[WARN] Dropped short samples (< target_T): {dropped_short}")
        print(f"[INFO] Total samples: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, start, label = self.samples[idx]

        x = np.load(path, mmap_mode="r")

        if start is None:
            # IND -> random crop (no full load)
            T = x.shape[-1]
            start = self.rng.integers(0, T - self.target_T + 1)

        x = x[:, :, start:start + self.target_T]

        return torch.from_numpy(x.copy()).float(), label

    def _fix_length(self, x, target_T=582):
        T = x.shape[-1]

        if T > target_T:
            start = 0
            x = x[:, :, start:start + target_T]

        return x

In [ ]:
def compute_target_T_from_npy(base_path, cases, max_samples=100):
    Ts = []

    for case in cases:
        normal_dir = os.path.join(base_path, "IND", case, "normal")
        paths = glob.glob(os.path.join(normal_dir, "*.npy"))

        for p in paths[:max_samples]:
            x = np.load(p)
            Ts.append(x.shape[-1])

    target_T = int(np.median(Ts))

    print(f"[INFO] target_T from IND: {target_T}")
    print(f"[INFO] min={min(Ts)}, max={max(Ts)}")

    return target_T

target_T = compute_target_T_from_npy(
    base_path=f"{DATA_BASE_PATH}/npy",
    cases=["case1", "case2", "case3", "case4"]
)

npy_dataset = UnifiedNPYDataset(
    base_path=f"{DATA_BASE_PATH}/npy",
    cases=["case1", "case2", "case3", "case4"],
    target_T=target_T, 
    use_ind=True,
    use_cnt=True,
    include_anomaly=True, 
    stride_ratio=5.0
)
print(len(npy_dataset), target_T)

In [ ]:
random.seed(42)

# -------------------------
# Separate indices
# -------------------------
normal_indices = [i for i, (_, _, y) in enumerate(npy_dataset.samples) if y == 0]
anom_indices   = [i for i, (_, _, y) in enumerate(npy_dataset.samples) if y == 1]

# split normal into IND / CNT
ind_indices = [i for i in normal_indices if npy_dataset.samples[i][1] is None]
cnt_indices = [i for i in normal_indices if npy_dataset.samples[i][1] is not None]

# -------------------------
# Shuffle separately
# -------------------------
random.shuffle(ind_indices)
random.shuffle(cnt_indices)

# -------------------------
# Split function
# -------------------------
def split_indices(indices):
    n = len(indices)
    train = indices[:int(0.8 * n)]
    val   = indices[int(0.8 * n):int(0.9 * n)]
    test  = indices[int(0.9 * n):]
    return train, val, test

# -------------------------
# Stratified split
# -------------------------
ind_train, ind_val, ind_test = split_indices(ind_indices)
cnt_train, cnt_val, cnt_test = split_indices(cnt_indices)

# merge back
train_idx        = ind_train + cnt_train
val_normal_idx   = ind_val + cnt_val
test_normal_idx  = ind_test + cnt_test

# shuffle merged splits
random.shuffle(train_idx)
random.shuffle(val_normal_idx)
random.shuffle(test_normal_idx)

# -------------------------
# ANOMALY split (50 / 50)
# -------------------------
random.shuffle(anom_indices)

m = len(anom_indices)
val_anom_idx  = anom_indices[:m // 2]
test_anom_idx = anom_indices[m // 2:]

# -------------------------
# DATASETS
# -------------------------
train_dataset = Subset(npy_dataset, train_idx)

val_normal = Subset(npy_dataset, val_normal_idx)
val_anom   = Subset(npy_dataset, val_anom_idx)

test_normal = Subset(npy_dataset, test_normal_idx)
test_anom   = Subset(npy_dataset, test_anom_idx)

In [ ]:
def count_ind_cnt(subset):
    ind_count = 0
    cnt_count = 0

    for idx in subset.indices:
        path, start, label = npy_dataset.samples[idx]

        if start is None:
            ind_count += 1
        else:
            cnt_count += 1

    return ind_count, cnt_count

train_ind, train_cnt = count_ind_cnt(train_dataset)
val_ind, val_cnt     = count_ind_cnt(val_normal)
test_ind, test_cnt   = count_ind_cnt(test_normal)

print("Train Normal  -> IND:", train_ind, "CNT:", train_cnt)
print("Val Normal    -> IND:", val_ind,   "CNT:", val_cnt)
print("Test Normal   -> IND:", test_ind,  "CNT:", test_cnt)

print(len(val_anom), len(test_anom))

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

vn_loader = DataLoader(val_normal, batch_size=8, shuffle=False)
va_loader = DataLoader(val_anom, batch_size=8, shuffle=False)

tn_loader = DataLoader(test_normal, batch_size=8, shuffle=False)
ta_loader = DataLoader(test_anom, batch_size=8, shuffle=False)

In [ ]:
# -------------------------
# CASE MAPPING
# -------------------------
def infer_case_from_path(path: str) -> str:
    parts = path.replace("\\", "/").split("/")
    if "IND" in parts:
        ind_pos = parts.index("IND")
        if ind_pos + 1 < len(parts):
            return parts[ind_pos + 1]
    return "unknown"

sample_cases = [infer_case_from_path(p) for (p, _, _) in npy_dataset.samples]

# =========================
# TRAIN LOADER (NORMAL ONLY)
# =========================
train_samples = [npy_dataset.samples[i] for i in train_idx]
labels = ["IND" if start is None else "CNT" for (_, start, _) in train_samples]

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

# =========================
# VALIDATION LOADERS
# =========================
def make_val_loaders(selected_cases, batch_size=8, anom_ratio=0.05):
    selected_cases = set(selected_cases)

    # -------------------------
    # NORMAL (VAL)
    # -------------------------
    sel_val_normal_idx = [
        i for i in val_normal_idx
        if sample_cases[i] in selected_cases
    ]

    # -------------------------
    # ANOMALY (VAL ONLY!)
    # -------------------------
    sel_val_anom_idx = [
        i for i in val_anom_idx
        if sample_cases[i] in selected_cases
    ]

    # -------------------------
    # Downsample anomalies
    # -------------------------
    n_normal = len(sel_val_normal_idx)
    target_anom = max(1, int(n_normal * anom_ratio))

    if len(sel_val_anom_idx) > target_anom:
        sel_val_anom_idx = random.sample(sel_val_anom_idx, target_anom)

    # -------------------------
    # Build subsets
    # -------------------------
    val_normal_subset = Subset(npy_dataset, sel_val_normal_idx)
    val_anom_subset   = Subset(npy_dataset, sel_val_anom_idx)

    val_normal_loader = DataLoader(val_normal_subset, batch_size=batch_size, shuffle=False)
    val_anom_loader   = DataLoader(val_anom_subset, batch_size=batch_size, shuffle=False)

    return val_normal_loader, val_anom_loader, len(sel_val_normal_idx), len(sel_val_anom_idx)

def make_test_loaders(selected_cases, batch_size=8, anom_ratio=0.05):
    selected_cases = set(selected_cases)

    # -------------------------
    # NORMAL (TEST)
    # -------------------------
    sel_test_normal_idx = [
        i for i in test_normal_idx
        if sample_cases[i] in selected_cases
    ]

    # -------------------------
    # ANOMALY (TEST)
    # -------------------------
    sel_test_anom_idx = [
        i for i in test_anom_idx
        if sample_cases[i] in selected_cases
    ]

    # -------------------------
    # Downsample anomalies (same as val)
    # -------------------------
    n_normal = len(sel_test_normal_idx)
    target_anom = max(1, int(n_normal * anom_ratio))

    if len(sel_test_anom_idx) > target_anom:
        sel_test_anom_idx = random.sample(sel_test_anom_idx, target_anom)

    # -------------------------
    # Build subsets
    # -------------------------
    test_normal_subset = Subset(npy_dataset, sel_test_normal_idx)
    test_anom_subset   = Subset(npy_dataset, sel_test_anom_idx)

    test_normal_loader = DataLoader(test_normal_subset, batch_size=batch_size, shuffle=False)
    test_anom_loader   = DataLoader(test_anom_subset, batch_size=batch_size, shuffle=False)

    return test_normal_loader, test_anom_loader, len(sel_test_normal_idx), len(sel_test_anom_idx)

# =========================
# CASE SCOPES
# =========================

VAL_CASE_SCOPES = {
    "case1_only": [CASES[0]],
    "case2_only": [CASES[1]],
    "case3_only": [CASES[2]],
    "case4_only": [CASES[3]],
    "all_cases": CASES,
}

# =========================
# BUILD VAL LOADERS
# =========================
val_loaders = {}

for scope_name, scope_cases in VAL_CASE_SCOPES.items():
    vn_loader, va_loader, n_normal, n_anom = make_val_loaders(scope_cases)

    val_loaders[scope_name] = {
        "val_normal_loader": vn_loader,
        "val_anom_loader": va_loader,
        "n_normal": n_normal,
        "n_anom": n_anom,
    }

    print(f"[VAL] {scope_name}: normal={n_normal}, anomaly={n_anom}")

test_loaders = {}

for scope_name, scope_cases in VAL_CASE_SCOPES.items():
    tn_loader, ta_loader, n_normal, n_anom = make_test_loaders(scope_cases)

    test_loaders[scope_name] = {
        "test_normal_loader": tn_loader,
        "test_anom_loader": ta_loader,
        "n_normal": n_normal,
        "n_anom": n_anom,
    }

    print(f"[TEST] {scope_name}: normal={n_normal}, anomaly={n_anom}")

val_normal_loader = val_loaders["all_cases"]["val_normal_loader"]
val_anom_loader   = val_loaders["all_cases"]["val_anom_loader"]
test_normal_loader = test_loaders["all_cases"]["test_normal_loader"]
test_anom_loader   = test_loaders["all_cases"]["test_anom_loader"]

## 6. Model

In [ ]:
class TransformerAutoEncoder(nn.Module):
    def __init__(
        self,
        input_shape=(4, 64, 64),
        time_patch=8,
        emb_dim=128,
        depth=3,
        num_heads=4,
        latent_dim=32,
        dropout=0.1
    ):
        super().__init__()

        C, H, W = input_shape
        self.input_shape = input_shape
        self.time_patch = time_patch

        self.patch_dim = C * H * time_patch

        # ---------------------
        # Patch embedding (time only)
        # ---------------------
        self.patch_embed = nn.Linear(self.patch_dim, emb_dim)

        self.pos_embed = nn.Parameter(torch.randn(1, 1000, emb_dim))  # max length buffer

        # ---------------------
        # Transformer encoder
        # ---------------------
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=emb_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        # ---------------------
        # Bottleneck
        # ---------------------
        self.fc_enc = nn.Linear(emb_dim, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, emb_dim)

        # ---------------------
        # Transformer decoder
        # ---------------------
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=emb_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.decoder = nn.TransformerEncoder(decoder_layer, num_layers=depth)

        # ---------------------
        # Reconstruction
        # ---------------------
        self.output_proj = nn.Linear(emb_dim, self.patch_dim)

    def patchify(self, x):
        B, C, H, W = x.shape
        p = self.time_patch

        W_new = (W // p) * p
        x = x[:, :, :, :W_new]

        x = x.unfold(3, p, p)  # (B, C, H, N, p)
        x = x.permute(0, 3, 1, 2, 4)  # (B, N, C, H, p)
        x = x.reshape(B, -1, self.patch_dim)

        return x, W_new

    def unpatchify(self, x, W_new):
        B, N, _ = x.shape
        p = self.time_patch
        C, H = self.input_shape[0], self.input_shape[1]

        x = x.view(B, N, C, H, p)
        x = x.permute(0, 2, 3, 1, 4)
        x = x.reshape(B, C, H, W_new)

        return x

    def forward(self, x):
        patches, W_new = self.patchify(x)

        N = patches.shape[1]

        z = self.patch_embed(patches)
        z = z + self.pos_embed[:, :N, :]

        z = self.encoder(z)

        z = self.fc_enc(z)
        z = self.fc_dec(z)

        z = self.decoder(z)

        patches = self.output_proj(z)

        out = self.unpatchify(patches, W_new)

        return F.interpolate(out, size=x.shape[-2:], mode='bilinear', align_corners=False)

## 7. Fine Tuning

In [ ]:
TUNE_SCOPE = "all_cases"
N_TRIALS = 30
TUNE_EPOCHS = 20

def objective(trial):
    print(f"\n=== Trial {trial.number} ===")

    # ======================
    # TRANSFORMER PARAMS
    # ======================
    time_patch = trial.suggest_categorical("time_patch", [4, 8, 16])
    emb_dim = trial.suggest_categorical("emb_dim", [64, 128])
    trans_depth = trial.suggest_int("trans_depth", 2, 4)
    num_heads = trial.suggest_categorical("num_heads", [2, 3])

    if emb_dim % num_heads != 0:
        raise optuna.TrialPruned()

    latent_dim = trial.suggest_categorical("latent_dim", [8, 16, 32])

    # ======================
    # TRAINING PARAMS
    # ======================
    lr = trial.suggest_float("lr", 1e-4, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.2)

    batch_size = trial.suggest_categorical("batch_size", [8, 16])
    noise_std = trial.suggest_float("noise_std", 0.0, 0.05)

    score_type = trial.suggest_categorical("score_type", ["l1", "l2"])

    # ======================
    # DATA
    # ======================
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    sample_x, _ = next(iter(train_loader))
    input_shape = sample_x.shape[1:]

    # ======================
    # MODEL
    # ======================
    model = TransformerAutoEncoder(
        input_shape=input_shape,
        time_patch=time_patch,
        emb_dim=emb_dim,
        depth=trans_depth,
        num_heads=num_heads,
        latent_dim=latent_dim,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    if score_type == "l1":
        criterion = nn.L1Loss()
    else:
        criterion = nn.MSELoss()

    # ======================
    # HELPER
    # ======================
    def reconstruction_errors(loader):
        model.eval()
        errors = []

        with torch.no_grad():
            for x, _ in loader:
                x = x.to(device)
                recon = model(x)

                if score_type == "l1":
                    err = torch.mean(torch.abs(x - recon), dim=(1,2,3))
                else:
                    err = torch.mean((x - recon)**2, dim=(1,2,3))

                errors.extend(err.cpu().numpy())

        return np.array(errors)

    def compute_pr_auc(vn_loader, va_loader):
        normal_scores = reconstruction_errors(vn_loader)
        anom_scores   = reconstruction_errors(va_loader)

        y_true = np.concatenate([
            np.zeros(len(normal_scores)),
            np.ones(len(anom_scores))
        ])
        y_score = np.concatenate([normal_scores, anom_scores])

        if len(np.unique(y_true)) < 2:
            return 0.5

        return average_precision_score(y_true, y_score), normal_scores, anom_scores

    # ======================
    # TRAIN
    # ======================
    for epoch in range(TUNE_EPOCHS):
        model.train()
        total_loss = 0

        for x, _ in train_loader:
            x = x.to(device)

            if noise_std > 0:
                x_noisy = x + noise_std * torch.randn_like(x)
            else:
                x_noisy = x

            recon = model(x_noisy)
            loss = criterion(recon, x)

            optimizer.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()

            total_loss += loss.item()

        if epoch % 5 == 0:
            vn_loader = val_loaders[TUNE_SCOPE]["val_normal_loader"]
            va_loader = val_loaders[TUNE_SCOPE]["val_anom_loader"]

            pr_auc, _, _ = compute_pr_auc(vn_loader, va_loader)

            if epoch >= 10:
                trial.report(pr_auc, epoch)
                if trial.should_prune():
                    raise optuna.TrialPruned()

    # ======================
    # FINAL VALIDATION
    # ======================
    results = {}

    for scope_name, loaders in val_loaders.items():
        vn_loader = loaders["val_normal_loader"]
        va_loader = loaders["val_anom_loader"]

        pr_auc, normal_scores, anom_scores = compute_pr_auc(vn_loader, va_loader)

        results[scope_name] = pr_auc
        trial.set_user_attr(f"{scope_name}_pr_auc", float(pr_auc))

        if scope_name == TUNE_SCOPE:
            gap = float(np.mean(anom_scores) - np.mean(normal_scores))
            trial.set_user_attr("gap", gap)

    print("Validation PR AUC:")
    for k, v in results.items():
        print(f"  {k}: {v:.4f}")

    return float(results[TUNE_SCOPE])

In [ ]:
optuna.logging.set_verbosity(optuna.logging.INFO)
def print_callback(study, trial):
    print(f"\nTrial {trial.number} finished")
    if trial.value is None:
        print("[ERROR] Failed!")
    else:
        print(f"  Value: {trial.value:.4f}")
    print(f"  Params: {trial.params}")

study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=N_TRIALS, callbacks=[print_callback])

for t in study.trials:
    print_callback(study, t)

print("[BEST] AUC:", study.best_value)
print("[BEST] params:", study.best_params)
print("[BEST] gap:", study.best_trial.user_attrs.get("gap", None))

In [ ]:

# ================================
# Optimization History
# ================================
# Shows how the objective value (PR-AUC) evolves over trials.
# Useful for checking:
# - whether the search is converging
# - whether more trials are needed
# - stability of the optimization process
vis.plot_optimization_history(study).show()


# ================================
# Hyperparameter Importance
# ================================
# Ranks hyperparameters based on their impact on the objective.
# Useful for:
# - identifying which parameters matter most (e.g., lr, latent_dim)
# - pruning irrelevant parts of the search space
vis.plot_param_importances(study).show()


## 8. Training the Best Model

In [ ]:
# Rebuild model with best params
best_params = study.best_params
time_patch=best_params['time_patch']
emb_dim=best_params['emb_dim']
trans_depth=best_params['trans_depth']
num_heads=best_params['num_heads']
latent_dim=best_params['latent_dim']
dropout=best_params['dropout']

sample_x, _ = next(iter(train_loader))
input_shape = sample_x.shape[1:]
model = TransformerAutoEncoder(
    input_shape=input_shape,
    time_patch=time_patch,
    emb_dim=emb_dim,
    depth=trans_depth,
    num_heads=num_heads,
    latent_dim=latent_dim,
    dropout=dropout
).to(device)

In [ ]:
# Train final model

train_loader = DataLoader(train_dataset, batch_size=best_params["batch_size"], shuffle=True)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=best_params["lr"],
    weight_decay=best_params["weight_decay"]
)

criterion = nn.L1Loss()

for epoch in range(TUNE_EPOCHS):
    model.train()
    for x, _ in train_loader:
        x = x.to(device)

        if best_params["noise_std"] > 0:
            x_noisy = x + best_params["noise_std"] * torch.randn_like(x)
        else:
            x_noisy = x

        recon = model(x_noisy)
        loss = criterion(recon, x)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)

## 9. Model Evaluation

In [ ]:
# Compute validation threshold (F1)

def get_scores(loader):
    model.eval()
    scores = []
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            recon = model(x)

            if best_params["score_type"] == "l1":
                err = torch.mean(torch.abs(x - recon), dim=(1,2,3))
            else:
                err = torch.mean((x - recon)**2, dim=(1,2,3))

            scores.extend(err.cpu().numpy())
    return np.array(scores)

# validation scores
val_normal_scores = get_scores(vn_loader)
val_anom_scores   = get_scores(va_loader)

y_true = np.concatenate([
    np.zeros(len(val_normal_scores)),
    np.ones(len(val_anom_scores))
])
y_score = np.concatenate([val_normal_scores, val_anom_scores])

precision, recall, thresholds = precision_recall_curve(y_true, y_score)
f1 = 2 * precision * recall / (precision + recall + 1e-8)

best_idx = np.argmax(f1[:-1])
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)

In [ ]:
# Evaluate on test

# model = CNN_AutoEncoder(
#     channels=channels,          # must match training
#     latent_dim=latent_dim,
#     input_shape=input_shape
# ).to(device)

# model.load_state_dict(torch.load("best_CNN_AE.pth", map_location=device))
model.eval()

test_normal_scores = get_scores(tn_loader)
test_anom_scores   = get_scores(ta_loader)

y_true_test = np.concatenate([
    np.zeros(len(test_normal_scores)),
    np.ones(len(test_anom_scores))
])
y_score_test = np.concatenate([test_normal_scores, test_anom_scores])

# -------------------------
# Downsample test anomalies
# -------------------------
anom_ratio = 0.05

n_normal = len(test_normal_scores)
target_anom = max(1, int(n_normal * anom_ratio))
print(target_anom)

if len(test_anom_scores) > target_anom:
    sampled_idx = random.sample(range(len(test_anom_scores)), target_anom)
    test_anom_scores = test_anom_scores[sampled_idx]

# -------------------------
# Recompute metrics
# -------------------------
y_true_test = np.concatenate([
    np.zeros(len(test_normal_scores)),
    np.ones(len(test_anom_scores))
])
y_score_test = np.concatenate([test_normal_scores, test_anom_scores])

pr_auc = average_precision_score(y_true_test, y_score_test)

y_pred = (y_score_test > best_threshold).astype(int)
f1 = f1_score(y_true_test, y_pred)


print("Best threshold:", best_threshold)
print("Test PR-AUC:", pr_auc)
print("Test F1:", f1)
print("Normal Mean Error:", test_normal_scores.mean())
print("Anomalous Mean Error:", test_anom_scores.mean())

In [ ]:
# Per case test results

print("\n=== Per-case Test Results ===")

all_normal_scores = []
all_anom_scores = []

anom_ratio = 1

for scope_name, loaders in test_loaders.items():
    tn_loader = loaders["test_normal_loader"]
    ta_loader = loaders["test_anom_loader"]

    normal_scores = get_scores(tn_loader)
    anom_scores   = get_scores(ta_loader)
    print(normal_scores.shape, anom_scores.shape)

    # -------------------------
    # Downsample anomalies (PER CASE)
    # -------------------------
    n_normal = len(normal_scores)
    target_anom = max(1, int(n_normal * anom_ratio))

    if len(anom_scores) > target_anom:
        sampled_idx = random.sample(range(len(anom_scores)), target_anom)
        anom_scores = anom_scores[sampled_idx]

    # -------------------------
    # Accumulate (skip all_cases)
    # -------------------------
    if scope_name != "all_cases":
        all_normal_scores.extend(normal_scores)
        all_anom_scores.extend(anom_scores)

    # -------------------------
    # Metrics
    # -------------------------
    y_true = np.concatenate([
        np.zeros(len(normal_scores)),
        np.ones(len(anom_scores))
    ])
    y_score = np.concatenate([normal_scores, anom_scores])

    pr_auc = average_precision_score(y_true, y_score)
    f1 = f1_score(y_true, (y_score > best_threshold).astype(int))
    gap = anom_scores.mean() - normal_scores.mean()

    print(f"{scope_name}: PR-AUC={pr_auc:.4f}, F1={f1:.4f}, Gap={gap:.4f}")

## 10. Result visualization

In [ ]:
def plot_multichannel_spec(x, title="Spectrogram", is_error=False):
    x = x.cpu().numpy()
    C = x.shape[0]

    fig, axs = plt.subplots(C, 1, figsize=(10, 2*C), sharex=True)

    if C == 1:
        axs = [axs]

    im = None
    for i in range(C):
        if is_error:
            vmin, vmax = 0, x[i].max()
        else:
            vmin, vmax = 0, 1

        im = axs[i].imshow(
            x[i],
            aspect='auto',
            origin='lower',
            cmap='magma',
            vmin=vmin,
            vmax=vmax
        )

        axs[i].set_ylabel(f"Ch {i+1}")
        axs[i].set_yticks([])

    axs[-1].set_xlabel("Time")
    fig.suptitle(title)

    plt.tight_layout(rect=[0, 0, 1.05, 1])
    fig.colorbar(im, ax=axs, fraction=0.02, pad=0.02)

    plt.show()

def plot_one_reconstruction(model, loader_iter, device):
    model.eval()

    x, label = next(loader_iter)
    x = x.to(device)

    with torch.no_grad():
        recon = model(x)

    x = x[0].cpu()
    recon = recon[0].cpu()

    # -------------------------
    # Compute error (L1)
    # -------------------------
    err = torch.abs(x - recon).mean().item()

    plot_multichannel_spec(x, "Original Log-Mel Spectrogram  (Randomly Sampled Test Data)")
    plot_multichannel_spec(
        recon,
        f"Reconstructed Log-Mel Spectrogram | Error: {err:.4f}"
    )
    plot_multichannel_spec(
        torch.abs(x - recon),
        f"Error Map | Mean Error: {err:.4f}", 
        is_error=True
    )

In [ ]:
ta_loader_iter = iter(ta_loader)
plot_one_reconstruction(model, ta_loader_iter, device) 

In [ ]:
tn_loader_iter = iter(tn_loader) 
plot_one_reconstruction(model, tn_loader_iter, device) 